# Section 2: Multiclass Network Intrusion Detection

This self-contained notebook downloads NSL-KDD, maps attacks to five classes, preprocesses the mixed data, and compares a Random Forest with a class-weighted 1D CNN. It does not import project source files or call repository scripts.

In [ ]:
%pip install -q joblib matplotlib numpy pandas scikit-learn torch

In [ ]:
import copy
import hashlib
import json
import os
import random
import shutil
import urllib.request
from dataclasses import dataclass
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def find_local_root() -> Path:
    current = Path.cwd().resolve()
    return next((p for p in [current, *current.parents] if (p / "pyproject.toml").is_file()), current)

RUNNING_ON_COLAB = Path("/content").is_dir()
WORK_DIR = Path("/content/section_02_workspace") if RUNNING_ON_COLAB else find_local_root()
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
config = {
    "seed": 42, "data_dir": "data/raw/nsl-kdd",
    "processed_dir": "data/processed/section_02",
    "models_dir": "models/section_02", "results_dir": "reports/section_02",
    "validation_fraction": 0.15, "selected_features": 64,
    "random_forest": {"n_estimators": 180, "max_depth": 28, "min_samples_leaf": 1, "max_features": "sqrt"},
    "cnn": {"dropout": 0.3, "batch_size": 512, "epochs": 8, "learning_rate": 0.001, "weight_decay": 0.0001, "class_weight_power": 0.5, "early_stopping_patience": 2},
}
random.seed(config["seed"])
np.random.seed(config["seed"])
print("Environment:", "Google Colab" if RUNNING_ON_COLAB else "Local")
print("Working directory:", WORK_DIR)
print("CUDA available:", torch.cuda.is_available())

## 1. Download and verify NSL-KDD

In [ ]:
NSL_BASE_URL = "https://raw.githubusercontent.com/HoaNP/NSL-KDD-DataSet/master"
NSL_FILES = {
    "KDDTrain+.txt": {"rows": 125973, "columns": 43, "sha256": "1b86d2f957b33082081bba410fe129b475efebcc13c9014c3f447c8271aadf95"},
    "KDDTest+.txt": {"rows": 22544, "columns": 43, "sha256": "fa46b0935342616aa83b7c2578db355b6a7aaabbc492248172c7a1e8b7ab8f84"},
}

def validate_nsl_file(path: Path, expected: dict[str, object]) -> None:
    digest, rows = hashlib.sha256(), 0
    with path.open("rb") as stream:
        for raw_line in stream:
            digest.update(raw_line)
            if raw_line.strip():
                rows += 1
                if rows == 1 and len(raw_line.decode("utf-8").rstrip().split(",")) != int(expected["columns"]):
                    raise ValueError(f"Unexpected column count in {path}")
    if rows != int(expected["rows"]) or digest.hexdigest() != expected["sha256"]:
        raise ValueError(f"Integrity check failed for {path}")

def download_nsl_kdd(destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    for filename, expected in NSL_FILES.items():
        path = destination / filename
        if not path.is_file():
            url = f"{NSL_BASE_URL}/{filename.replace('+', '%2B')}"
            print("Downloading:", url)
            urllib.request.urlretrieve(url, path)
        validate_nsl_file(path, expected)
        print("Verified:", path)

download_nsl_kdd(Path(config["data_dir"]))

## 2. Load labels and create train, validation, and test partitions

In [ ]:
FEATURE_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login",
    "count", "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
]
CATEGORICAL_COLUMNS = ["protocol_type", "service", "flag"]
NUMERIC_COLUMNS = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_COLUMNS]
DATA_COLUMNS = [*FEATURE_COLUMNS, "attack_name", "difficulty"]
CLASS_NAMES = ["normal", "dos", "probe", "r2l", "u2r"]
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
ATTACK_CATEGORY = {
    **{name: "dos" for name in "back land neptune pod smurf teardrop mailbomb apache2 processtable udpstorm".split()},
    **{name: "probe" for name in "ipsweep nmap portsweep satan saint mscan".split()},
    **{name: "r2l" for name in "ftp_write guess_passwd imap multihop phf spy warezclient warezmaster sendmail named snmpgetattack snmpguess xlock xsnoop worm".split()},
    **{name: "u2r" for name in "buffer_overflow loadmodule perl rootkit ps sqlattack xterm httptunnel".split()},
    "normal": "normal",
}

@dataclass(frozen=True)
class DatasetPartitions:
    train: pd.DataFrame
    validation: pd.DataFrame
    test: pd.DataFrame

def map_attack_category(name: str) -> str:
    normalized = str(name).strip().lower().removesuffix(".")
    if normalized not in ATTACK_CATEGORY:
        raise ValueError(f"Unknown attack label: {name!r}")
    return ATTACK_CATEGORY[normalized]

def load_nsl_partition(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, names=DATA_COLUMNS, header=None)
    frame["attack_name"] = frame["attack_name"].astype(str).str.strip().str.lower().str.rstrip(".")
    frame["label_name"] = frame["attack_name"].map(map_attack_category)
    frame["label"] = frame["label_name"].map(CLASS_TO_INDEX).astype("int64")
    return frame

def load_nsl_kdd(data_dir: Path, validation_fraction: float, seed: int) -> DatasetPartitions:
    full_train = load_nsl_partition(data_dir / "KDDTrain+.txt")
    test = load_nsl_partition(data_dir / "KDDTest+.txt")
    train, validation = train_test_split(
        full_train, test_size=validation_fraction, random_state=seed, stratify=full_train["label"],
    )
    return DatasetPartitions(train.reset_index(drop=True), validation.reset_index(drop=True), test.reset_index(drop=True))

def profile_partition(frame: pd.DataFrame) -> dict[str, object]:
    return {
        "records": int(len(frame)), "missing_values": int(frame[DATA_COLUMNS].isna().sum().sum()),
        "duplicate_records": int(frame[DATA_COLUMNS].duplicated().sum()),
        "class_counts": {name: int((frame["label_name"] == name).sum()) for name in CLASS_NAMES},
    }

partitions = load_nsl_kdd(Path(config["data_dir"]), config["validation_fraction"], config["seed"])
profiles = {name: profile_partition(frame) for name, frame in {"train": partitions.train, "validation": partitions.validation, "test": partitions.test}.items()}
display(pd.DataFrame([{"split": name, "records": p["records"], **p["class_counts"]} for name, p in profiles.items()]))

## 3. Leakage-safe preprocessing and evaluation

In [ ]:
def build_feature_pipeline(selected_features: int) -> Pipeline:
    numerical = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    categorical = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    columns = ColumnTransformer([("numeric", numerical, NUMERIC_COLUMNS), ("categorical", categorical, CATEGORICAL_COLUMNS)], verbose_feature_names_out=False)
    return Pipeline([("columns", columns), ("variance", VarianceThreshold()), ("selection", SelectKBest(f_classif, k=selected_features))])

def fit_transform_features(pipeline: Pipeline, train: pd.DataFrame, validation: pd.DataFrame, test: pd.DataFrame):
    values = (
        pipeline.fit_transform(train[FEATURE_COLUMNS], train["label"]),
        pipeline.transform(validation[FEATURE_COLUMNS]), pipeline.transform(test[FEATURE_COLUMNS]),
    )
    return tuple(np.asarray(value, dtype=np.float32) for value in values)

def selected_feature_names(pipeline: Pipeline) -> list[str]:
    names = pipeline.named_steps["columns"].get_feature_names_out()
    names = names[pipeline.named_steps["variance"].get_support()]
    return names[pipeline.named_steps["selection"].get_support()].tolist()

def multiclass_metrics(labels: np.ndarray, probabilities: np.ndarray) -> dict[str, object]:
    predictions = probabilities.argmax(axis=1)
    indices = np.arange(len(CLASS_NAMES))
    binary = label_binarize(labels, classes=indices)
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_precision": float(precision_score(labels, predictions, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(labels, predictions, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
        "macro_average_precision": float(average_precision_score(binary, probabilities, average="macro")),
        "confusion_matrix": confusion_matrix(labels, predictions, labels=indices).tolist(),
        "classification_report": classification_report(labels, predictions, labels=indices, target_names=CLASS_NAMES, output_dict=True, zero_division=0),
    }

def save_metrics(metrics: dict[str, object], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

def save_multiclass_plots(labels: np.ndarray, probabilities: np.ndarray, model_name: str, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    slug = model_name.lower().replace(" ", "-")
    predictions = probabilities.argmax(axis=1)
    figure, axis = plt.subplots(figsize=(7.2, 6))
    ConfusionMatrixDisplay.from_predictions(labels, predictions, labels=np.arange(len(CLASS_NAMES)), display_labels=CLASS_NAMES, cmap="Blues", colorbar=False, ax=axis)
    axis.set_title(f"{model_name}: confusion matrix")
    figure.tight_layout(); figure.savefig(output_dir / f"{slug}-confusion-matrix.png", dpi=180); plt.close(figure)
    binary = label_binarize(labels, classes=np.arange(len(CLASS_NAMES)))
    figure, axis = plt.subplots(figsize=(7.2, 5.4))
    for index, name in enumerate(CLASS_NAMES):
        precision, recall, _ = precision_recall_curve(binary[:, index], probabilities[:, index])
        ap = average_precision_score(binary[:, index], probabilities[:, index])
        axis.plot(recall, precision, label=f"{name} (AP={ap:.3f})")
    axis.set(xlabel="Recall", ylabel="Precision", title=f"{model_name}: precision-recall curves")
    axis.grid(alpha=0.25); axis.legend(loc="lower left"); figure.tight_layout()
    figure.savefig(output_dir / f"{slug}-precision-recall-curves.png", dpi=180); plt.close(figure)

feature_pipeline = build_feature_pipeline(config["selected_features"])
train_features, validation_features, test_features = fit_transform_features(feature_pipeline, partitions.train, partitions.validation, partitions.test)
feature_names = selected_feature_names(feature_pipeline)
train_labels = partitions.train["label"].to_numpy(dtype=np.int64)
validation_labels = partitions.validation["label"].to_numpy(dtype=np.int64)
test_labels = partitions.test["label"].to_numpy(dtype=np.int64)
models_dir, results_dir = Path(config["models_dir"]), Path(config["results_dir"])
print(train_features.shape, validation_features.shape, test_features.shape)

## 4. Random Forest

In [ ]:
def train_random_forest(features: np.ndarray, labels: np.ndarray, model_config: dict[str, object], seed: int) -> RandomForestClassifier:
    model = RandomForestClassifier(
        n_estimators=int(model_config["n_estimators"]), max_depth=int(model_config["max_depth"]),
        min_samples_leaf=int(model_config["min_samples_leaf"]), max_features=str(model_config["max_features"]),
        class_weight="balanced_subsample", n_jobs=-1, random_state=seed,
    )
    model.fit(features, labels)
    return model

forest = train_random_forest(train_features, train_labels, config["random_forest"], config["seed"])
forest_probabilities = forest.predict_proba(test_features)
forest_metrics = multiclass_metrics(test_labels, forest_probabilities)
save_metrics(forest_metrics, results_dir / "metrics/random-forest.json")
save_multiclass_plots(test_labels, forest_probabilities, "Random Forest", results_dir / "figures")
models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump({"feature_pipeline": feature_pipeline, "model": forest, "class_names": CLASS_NAMES, "selected_features": feature_names}, models_dir / "random-forest.joblib")
display(pd.DataFrame(forest_metrics["classification_report"]).T)

## 5. Class-weighted 1D CNN

In [ ]:
class FeatureCNN(nn.Module):
    def __init__(self, classes: int, dropout: float) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveMaxPool1d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(64, classes),
        )
    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return self.network(values.unsqueeze(1))

def train_and_evaluate_cnn(train_features, train_labels, validation_features, validation_labels, test_features, test_labels, model_config, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    def make_loader(features, labels, shuffle):
        dataset = TensorDataset(torch.from_numpy(features), torch.from_numpy(labels.astype(np.int64)))
        return DataLoader(dataset, batch_size=int(model_config["batch_size"]), shuffle=shuffle, generator=torch.Generator().manual_seed(seed) if shuffle else None)
    train_loader = make_loader(train_features, train_labels, True)
    validation_loader = make_loader(validation_features, validation_labels, False)
    test_loader = make_loader(test_features, test_labels, False)
    counts = np.bincount(train_labels, minlength=len(CLASS_NAMES)).astype(float)
    weights = (len(train_labels) / (len(CLASS_NAMES) * counts)) ** float(model_config["class_weight_power"]); weights /= weights.mean()
    model = FeatureCNN(len(CLASS_NAMES), float(model_config["dropout"])).to(device)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device))
    optimizer = torch.optim.Adam(model.parameters(), lr=float(model_config["learning_rate"]), weight_decay=float(model_config["weight_decay"]))
    def evaluate(loader):
        model.eval(); total, probabilities = 0.0, []
        with torch.no_grad():
            for values, labels in loader:
                values, labels = values.to(device), labels.to(device)
                logits = model(values); total += criterion(logits, labels).item() * len(labels)
                probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
        return total / len(loader.dataset), np.concatenate(probabilities)
    history, best_state, best_f1, stale = [], copy.deepcopy(model.state_dict()), -1.0, 0
    for epoch in range(1, int(model_config["epochs"]) + 1):
        model.train(); training_loss = 0.0
        for values, labels in train_loader:
            values, labels = values.to(device), labels.to(device)
            optimizer.zero_grad(); loss = criterion(model(values), labels); loss.backward(); optimizer.step()
            training_loss += loss.item() * len(labels)
        training_loss /= len(train_loader.dataset)
        validation_loss, probabilities = evaluate(validation_loader)
        validation_f1 = float(multiclass_metrics(validation_labels, probabilities)["macro_f1"])
        history.append({"epoch": epoch, "training_loss": training_loss, "validation_loss": validation_loss, "validation_macro_f1": validation_f1})
        print(f"Epoch {epoch}: loss={training_loss:.4f}, validation_macro_f1={validation_f1:.4f}")
        if validation_f1 > best_f1:
            best_f1, best_state, stale = validation_f1, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
            if stale >= int(model_config["early_stopping_patience"]): break
    model.load_state_dict(best_state)
    _, probabilities = evaluate(test_loader)
    metrics = multiclass_metrics(test_labels, probabilities)
    metrics.update({"device": str(device), "best_validation_macro_f1": best_f1, "epochs_completed": len(history)})
    save_metrics(metrics, results_dir / "metrics/cnn.json")
    save_multiclass_plots(test_labels, probabilities, "1D CNN", results_dir / "figures")
    figure, axes = plt.subplots(1, 2, figsize=(10.5, 4.2)); epochs = [row["epoch"] for row in history]
    axes[0].plot(epochs, [row["training_loss"] for row in history], label="Train"); axes[0].plot(epochs, [row["validation_loss"] for row in history], label="Validation"); axes[0].legend(); axes[0].set_title("CNN loss")
    axes[1].plot(epochs, [row["validation_macro_f1"] for row in history]); axes[1].set_title("Validation macro F1"); figure.tight_layout()
    (results_dir / "figures").mkdir(parents=True, exist_ok=True); figure.savefig(results_dir / "figures/cnn-training-history.png", dpi=180); plt.close(figure)
    torch.save({"state_dict": model.state_dict(), "class_names": CLASS_NAMES, "config": model_config}, models_dir / "cnn.pt")
    return metrics

CNN_EPOCHS_OVERRIDE = None  # Change to 1 for a quick smoke run.
cnn_config = config["cnn"].copy()
if CNN_EPOCHS_OVERRIDE is not None: cnn_config["epochs"] = CNN_EPOCHS_OVERRIDE
cnn_metrics = train_and_evaluate_cnn(train_features, train_labels, validation_features, validation_labels, test_features, test_labels, cnn_config, config["seed"])
display(pd.DataFrame(cnn_metrics["classification_report"]).T)

## 6. Compare and export

In [ ]:
metric_names = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "macro_average_precision"]
comparison = pd.DataFrame([
    {"model": "Random Forest", **{name: forest_metrics[name] for name in metric_names}},
    {"model": "1D CNN", **{name: cnn_metrics[name] for name in metric_names}},
])
results_dir.mkdir(parents=True, exist_ok=True); comparison.to_csv(results_dir / "model-comparison.csv", index=False)
Path(config["processed_dir"]).mkdir(parents=True, exist_ok=True)
Path(config["processed_dir"]).joinpath("selected-features.json").write_text(json.dumps(feature_names, indent=2), encoding="utf-8")
display(comparison.style.format(precision=4))
if RUNNING_ON_COLAB:
    export = WORK_DIR / "section_02_export"
    if export.exists(): shutil.rmtree(export)
    shutil.copytree(results_dir, export / "reports"); shutil.copytree(models_dir, export / "models")
    print("Download:", shutil.make_archive("/content/section_02_results", "zip", root_dir=export))

## Limitations

NSL-KDD is a refined historical benchmark rather than raw contemporary traffic. Accuracy can hide weak R2L and U2R detection, so macro metrics and class-wise confusion must drive the comparison.